In [3]:
import os
from pathlib import Path
from ase import io
from ase.build import add_vacuum
from ase.optimize import BFGS
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.io.trajectory import Trajectory
from dftd4.ase import DFTD4

#adsorbates = ["Li2S", "Li2S2", "Li2S4", "Li2S6", "Li2S8", "S8"]
adsorbates = ["S8"]
run_root = Path(".")
run_root.mkdir(exist_ok=True)
pseudo_dir = Path("/home/ameer_ubuntu/Git_projects/QE/PP_NC")
vacuum = 7.5
ecutwfc = 60.0
ecutrho = 4 * ecutwfc
kpts = (1, 1, 1)

# Convergence thresholds converted from Rydberg to eV
# 1 Ry = 13.605693 eV, 1 Ry/Bohr = 25.71104 eV/Angstrom
etot_conv_thr_ry = 1.0e-4  # Ry
forc_conv_thr_ry = 5.0e-4  # Ry/Bohr

# Convert to eV units for ASE
etot_conv_thr = etot_conv_thr_ry * 13.605693  # eV
fmax = forc_conv_thr_ry * 25.71104  # eV/Angstrom

pseudos = {"Li": "Li_ONCV_PBE-1.2.upf", "S": "S_ONCV_PBE-1.2.upf"}

for ads in adsorbates:
    structure_path = Path(f"input_ads/{ads}_final.extxyz")
    run_dir = run_root / ads / 'D4'
    run_dir.mkdir(parents=True, exist_ok=True)

    atoms = io.read(structure_path)
    add_vacuum(atoms, vacuum)
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    profile = EspressoProfile(
        command="mpirun -n 16 /home/ameer_ubuntu/miniforge3/envs/qe/bin/pw.x", 
        pseudo_dir=str(pseudo_dir)
    )
    
    # Set up QE calculator for SCF
    qe_calc = Espresso(
        profile=profile,
        pseudopotentials=pseudos,
        input_data={
            "control": {
                "calculation": "scf",  # SCF for ASE optimization
                "prefix": ads,
                'tprnfor': True,
            },
            "system": {
                "ecutwfc": ecutwfc, 
                "ecutrho": ecutrho,
                "input_dft": "rpbe",    
                "occupations": "smearing",
                "smearing": "gaussian",
                "degauss": 0.001,
            },
            "electrons": {
                "conv_thr": 1.0e-6,
                "mixing_beta": 0.60,
                "electron_maxstep": 140,
                "mixing_mode": "local-TF",
            },
        },
        kpts=kpts,
        pseudo_dir=str(pseudo_dir),
        directory=str(run_dir),
    )
    
    # Combine QE and DFTD4 calculators
    atoms.calc = DFTD4(method="rpbe").add_calculator(qe_calc)
    
    # Run ASE optimization with BFGS
    opt = BFGS(atoms,logfile=str(run_dir / f"{ads}_relax.log"))
    traj = Trajectory(str(run_dir / f"{ads}.traj"), 'w', atoms)
    opt.attach(traj)
    opt.run(fmax=fmax)

    # Save final optimized structure to a clean .extxyz
    output_path = run_dir / f"{ads}.relaxed.extxyz"
    io.write(output_path, atoms, format="extxyz")
    
    print(f"Relaxation complete for {ads}. Structure -> {output_path}")


KeyboardInterrupt: 